# Hands-On Lab 1: Predictive AI & Machine Learning in Google Colab
### Predicting Natural Rubber Compound Properties from the Recipe — Vibe Coding Edition



## The Problem
A rubber products plant is developing a carbon-black-filled **natural rubber (NR)** compound. Every lab trial costs a day: weigh, mix on the two-roll mill, run the rheometer (MDR), press-cure, condition and test. The lab has **120 past trials**. Can a model learn from them which recipe gives the best tensile strength, before we mix the next batch?

**Fixed in every trial (phr = parts per hundred rubber):** NR 100 · ZnO 5 · stearic acid 2 · antioxidant 1  
**Varied:** sulphur, CBS accelerator, N330 carbon black, cure temperature


### 💡 What is "Vibe Coding"?
You describe your engineering intent in plain language; an AI assistant writes the code; **you** check whether the result makes physical sense.


### Step 1: Load the Lab Trials (3 min)

| Column | Meaning | Role |
| :--- | :--- | :--- |
| `Sulphur_phr` | Sulphur (curative), 0.5 – 3.0 phr | Input |
| `CBS_phr` | CBS accelerator, 0.5 – 2.0 phr | Input |
| `Carbon_Black_N330_phr` | Reinforcing filler, 0 – 60 phr | Input |
| `Cure_Temp_C` | Rheometer / press cure temperature, 140 – 170 °C | Input |
| `t90_min` | Time to 90 % cure (MDR) | Measured after |
| `Delta_Torque_dNm` | MDR torque rise M<sub>H</sub> − M<sub>L</sub> (crosslink indicator) | Measured after |
| `Hardness_ShoreA` | Hardness of the cured sheet | Measured after |
| `Tensile_Strength_MPa` | Tensile strength of the cured dumbbells | **Main target** |

> 💬 **Vibe Coding Prompt:**  
> *"Load data from https://raw.githubusercontent.com/dissabnd/ITUM2-IntroAI/refs/heads/main/data/rubber_compound_lab_data.csv with pandas. Print the number of trials, show the first 5 rows, and show the min, mean and max of every numeric column."*

### Step 2: Look Before You Model (4 min)
Left: tensile strength against sulphur, coloured by carbon black loading. Right: hardness against carbon black.

> 💬 **Vibe Coding Prompt:**  
> *"Make two plots side by side. Left: scatter of `Sulphur_phr` (x) vs `Tensile_Strength_MPa` (y), coloured by `Carbon_Black_N330_phr` with a colour bar. Right: scatter of `Carbon_Black_N330_phr` (x) vs `Hardness_ShoreA` (y). Add axis labels with units, titles and a grid."*

✍️ **Question 2:** Hardness rises in an almost straight line with carbon black. Does tensile strength keep rising as sulphur increases? What happens to a rubber network that is crosslinked too much?

### Step 3: Train Two Models for Tensile Strength (5 min)
- **Inputs ($X$):** the 4 recipe and cure variables.
- **Target ($y$):** `Tensile_Strength_MPa`.
- **Split:** train on 96 trials, test on 24 unseen trials.
- **Models:** a **straight-line (linear) regression** and a **Random Forest**.

> ⚠️ **Data leakage:** we do **not** use `t90_min`, `Delta_Torque_dNm` or `Hardness_ShoreA` as inputs. They are measured *after* the compound is mixed and cured, so they are not available when we choose a recipe.

> 💬 **Vibe Coding Prompt:**  
> *"Predict `Tensile_Strength_MPa` from `Sulphur_phr`, `CBS_phr`, `Carbon_Black_N330_phr` and `Cure_Temp_C`. Split 80/20 with random_state=42. Train a LinearRegression and a RandomForestRegressor (200 trees, min_samples_leaf=2, random_state=42). Print R² and mean absolute error on the test set for both, and draw predicted vs. measured plots side by side with a y = x line."*

### Step 4: Can We Trust Those Scores? (3 min)
24 test trials is a small sample. **5-fold cross-validation** tests every trial once and reports an average and a spread.

> 💬 **Vibe Coding Prompt:**  
> *"Run 5-fold cross-validation (shuffle=True, random_state=0) on both models using R² as the score. Print the mean and standard deviation for each."*

✍️ **Question 4:** How do the cross-validated scores compare with the single test-set scores in Step 3? Which numbers would you put in your lab report?

### Step 5: What Drives Each Property? (5 min)
Train one Random Forest per property and compare which inputs each one relies on.

> 💬 **Vibe Coding Prompt:**  
> *"For each of `Tensile_Strength_MPa`, `Hardness_ShoreA` and `t90_min`, train a RandomForestRegressor on the 4 inputs and plot the feature importances as a grouped horizontal bar chart. Then compare linear regression and Random Forest on hardness using 5-fold cross-validated R²."*

✍️ **Question 5:**
1. Which input controls **how fast** the compound cures (t90)? Which inputs control **how strong** it is?
2. For **hardness**, the simple straight-line model does as well as, or better than, the Random Forest. Why? (Look back at the right-hand plot in Step 2.)

### Step 6: Use the Model as a Formulator, and Test Its Limits (5 min)
**You are the compound development technologist.** Hold CBS at 1.0 phr, black at 40 phr and cure at 150 °C. Sweep sulphur and ask both models for the tensile strength.

> 💬 **Vibe Coding Prompt:**  
> *"Create a grid of sulphur from 0.5 to 3.0 phr with CBS 1.0 phr, carbon black 40 phr and cure temperature 150 °C. Predict tensile strength with both models and plot both curves. Print the sulphur level where the Random Forest predicts the highest tensile strength."*

Now try your own recipe. Run it at least twice: once inside the trial range, and once with **sulphur = 5.0 phr** (far outside the 0.5 – 3.0 phr the lab has tested).

> 💬 **Vibe Coding Prompt:**  
> *"Predict the tensile strength of one new recipe with the Random Forest. Before predicting, check every input against the min and max of the training data and print a clear extrapolation warning if any input is outside that range."*

✍️ **Question 6:** At 5.0 phr sulphur the Random Forest gives almost the same answer as at 3.0 phr. Given the trend you saw in the sweep, what would you expect a real 5 phr compound to do? What would you do before releasing this recipe?

---
## 💡 Key Takeaways
- **Look before you model:** hardness is close to linear in carbon black; tensile strength has an **optimum** in sulphur.
- **Match the model to the physics:** the straight-line model reports that sulphur barely matters, because the rise and the fall cancel. The Random Forest finds the optimum. For hardness, the simple model is enough.
- **Small test sets flatter:** report cross-validated scores, with their spread.
- **Different properties, different drivers:** cure temperature sets the **speed** of cure (t90); the recipe (black, sulphur) sets the **properties**.
- **Models do not extrapolate:** outside the tested range the model still returns a confident number. The technologist must notice and run a lab trial.
- **Predictive vs. generative AI:** here we learned from numbers to predict a property. In Labs 2 and 3, generative AI reasons in language and needs the same checking.